In [1]:
from pynq import Clocks

print("Avant :", Clocks.fclk0_mhz, "MHz")

Clocks.fclk0_mhz = 50.0

print("Après :", Clocks.fclk0_mhz, "MHz")

Avant : 50.0 MHz
Après : 50.0 MHz


In [2]:
from pathlib import Path

print("AXI masters:")
for p in Path("/sys/class").glob("*"):
    if "fpga" in p.name or "remoteproc" in p.name:
        print(" -", p)

AXI masters:
 - /sys/class/fpga_manager
 - /sys/class/fpga_bridge
 - /sys/class/fpga_region
 - /sys/class/remoteproc


In [3]:
from pynq import MMIO

slcr = MMIO(0xF8000000, 0x1000)

# Registre de contrôle des level shifters PS-PL
level_shifters = slcr.read(0x900)

# Registre de reset de la logique programmable
fpga_reset = slcr.read(0x240)

print(f"LVL_SHFTR_EN: 0x{level_shifters:08X}")
print(f"FPGA_RST_CTRL: 0x{fpga_reset:08X}")

LVL_SHFTR_EN: 0x0000000F
FPGA_RST_CTRL: 0x00000000


In [4]:
from pynq import MMIO

gpio_cmd = MMIO(0x41200000, 0x1000)

direction = gpio_cmd.read(0x04)

print(f"GPIO command direction: 0x{direction:08X}")

GPIO command direction: 0xFFFFFFFF


In [5]:
# AXI GPIO 0 : comand

# Canal 1 : 32 bits out (command)
gpio_cmd.write(0x04, 0x00000000)

# Canal 2 : 2 bits out (start and ack)
gpio_cmd.write(0x0C, 0x00000000)

# test
print(f"GPIO CH1 TRI: 0x{gpio_cmd.read(0x04):08X}")
print(f"GPIO CH2 TRI: 0x{gpio_cmd.read(0x0C):08X}")

GPIO CH1 TRI: 0xFFFFFFFF
GPIO CH2 TRI: 0xFFFFFFFF


In [6]:
from pynq import MMIO

gpio_cmd = MMIO(0x41200000, 0x1000)

# Garder START et ACK à zéro
gpio_cmd.write(0x08, 0x00000000)

# Écrire un motif de test dans command[31:0]
gpio_cmd.write(0x00, 0x00000203)

# Relire le registre de données
value = gpio_cmd.read(0x00)

print(f"GPIO_DATA: 0x{value:08X}")

GPIO_DATA: 0x00000203


In [7]:
from pynq import MMIO

gpio_status = MMIO(0x41210000, 0x1000)

response = gpio_status.read(0x00)
status = gpio_status.read(0x08)

print(f"ALU response: 0x{response:08X}")
print(f"ALU status:   0x{status:08X}")

print("Busy:", status & 0x01)
print("Done:", (status >> 1) & 0x01)

ALU response: 0x00000005
ALU status:   0x00000000
Busy: 0
Done: 0


In [8]:
import time

# AXI GPIO 0 : commandes
# AXI GPIO 1 : résultats

DATA = 0x00
DATA2 = 0x08

a = 3
b = 2
op = 0  # ADD

command = a | (b << 8) | (op << 16)

print(f"Command: 0x{command:08X}")

# Initialiser les signaux de contrôle
gpio_cmd.write(DATA2, 0x00)

# Envoyer les opérandes
gpio_cmd.write(DATA, command)

# Générer une impulsion START
gpio_cmd.write(DATA2, 0x01)
gpio_cmd.write(DATA2, 0x00)

# Attendre DONE, avec délai maximal
deadline = time.monotonic() + 1.0

while True:
    status = gpio_status.read(DATA2)

    if status & 0x02:
        break

    if time.monotonic() >= deadline:
        raise TimeoutError(
            f"ALU timeout, status=0x{status:08X}"
        )

# Récupérer le résultat
response = gpio_status.read(DATA)

result = response & 0xFF
carry = (response >> 8) & 1
borrow = (response >> 9) & 1
zero = (response >> 10) & 1
overflow = (response >> 11) & 1

print(f"Response: 0x{response:08X}")
print(f"Result: {result}")
print(f"Carry: {carry}")
print(f"Borrow: {borrow}")
print(f"Zero: {zero}")
print(f"Overflow: {overflow}")

# Acquitter la fin de l'opération
gpio_cmd.write(DATA2, 0x02)
gpio_cmd.write(DATA2, 0x00)

assert result == 5, f"Expected 5, got {result}"

Command: 0x00000203
Response: 0x00000005
Result: 5
Carry: 0
Borrow: 0
Zero: 0
Overflow: 0


In [10]:

import time

OPERATIONS = {
    "ADD": 0b000,
    "AND": 0b001,
    "OR":  0b010,
    "SUB": 0b011,
}

DATA = 0x00
CONTROL = 0x08

BUSY = 0b01
DONE = 0b10
START = 0b01
ACK = 0b10


def read_status():
    return gpio_status.read(CONTROL) & 0b11


def wait_status(predicate, timeout=1.0):
    deadline = time.monotonic() + timeout

    while True:
        status = read_status()

        if predicate(status):
            return status

        if time.monotonic() >= deadline:
            raise TimeoutError(
                f"ALU timeout: status=0b{status:02b}"
            )


def execute_alu(a, b, op):
    if not isinstance(a, int) or not isinstance(b, int):
        raise TypeError("Operands must be integers")

    if not (0 <= a <= 255 and 0 <= b <= 255):
        raise ValueError("Operands must be 8-bit unsigned integers")

    opcode = OPERATIONS[op.upper()]
    command = a | (b << 8) | (opcode << 16)

    # Ensure controls are inactive.
    gpio_cmd.write(CONTROL, 0)

    # Confirm controller is idle.
    wait_status(lambda s: s == 0)

    # Send the command.
    gpio_cmd.write(DATA, command)

    # Generate START pulse.
    gpio_cmd.write(CONTROL, START)
    gpio_cmd.write(CONTROL, 0)

    # Wait for operation completion.
    wait_status(lambda s: bool(s & DONE))

    response = gpio_status.read(DATA)

    # Acknowledge completed operation.
    gpio_cmd.write(CONTROL, ACK)
    gpio_cmd.write(CONTROL, 0)

    # Ensure DONE is cleared before another command.
    wait_status(lambda s: s == 0)

    return {
        "result": response & 0xFF,
        "carry": (response >> 8) & 1,
        "borrow": (response >> 9) & 1,
        "zero": (response >> 10) & 1,
        "overflow": (response >> 11) & 1,
    }


In [12]:
result = execute_alu(4, 2, "ADD")

print(result)

{'result': 6, 'carry': 0, 'borrow': 0, 'zero': 0, 'overflow': 0}


In [13]:
tests = [
    ("ADD", 3, 2, 5),
    ("SUB", 7, 3, 4),
    ("AND", 0b1100, 0b1010, 8),
    ("OR",  0b1100, 0b1010, 14),
]

for op, a, b, expected in tests:
    actual = execute_alu(a, b, op)["result"]

    print(
        f"{op}: {a}, {b} -> {actual} "
        f"(expected {expected})",
        flush=True
    )

    assert actual == expected

print("All hardware ALU tests passed!")

ADD: 3, 2 -> 5 (expected 5)
SUB: 7, 3 -> 4 (expected 4)
AND: 12, 10 -> 8 (expected 8)
OR: 12, 10 -> 14 (expected 14)
All hardware ALU tests passed!


In [14]:
tests = [
    ("ADD", 255, 1,   0,   "carry"),
    ("SUB", 3,   5,   254, "borrow"),
    ("SUB", 7,   7,   0,   "zero"),
    ("ADD", 127, 1,   128, "overflow"),
    ("SUB", 128, 1,   127, "overflow"),
]

for op, a, b, expected, flag in tests:
    response = execute_alu(a, b, op)

    assert response["result"] == expected, (
        f"{op}({a}, {b}): unexpected result {response}"
    )
    assert response[flag] == 1, (
        f"{op}({a}, {b}): expected {flag}=1, got {response}"
    )

    print(f"PASS {op}({a}, {b}) -> {response}", flush=True)

print("All ALU flag tests passed!")

PASS ADD(255, 1) -> {'result': 0, 'carry': 1, 'borrow': 0, 'zero': 1, 'overflow': 0}
PASS SUB(3, 5) -> {'result': 254, 'carry': 0, 'borrow': 1, 'zero': 0, 'overflow': 0}
PASS SUB(7, 7) -> {'result': 0, 'carry': 0, 'borrow': 0, 'zero': 1, 'overflow': 0}
PASS ADD(127, 1) -> {'result': 128, 'carry': 0, 'borrow': 0, 'zero': 0, 'overflow': 1}
PASS SUB(128, 1) -> {'result': 127, 'carry': 0, 'borrow': 0, 'zero': 0, 'overflow': 1}
All ALU flag tests passed!


In [15]:
def reference_add(a, b):
    total = a + b
    result = total & 0xFF

    return {
        "result": result,
        "carry": int(total > 255),
        "zero": int(result == 0),
        "overflow": int(
            ((~(a ^ b) & (a ^ result)) & 0x80) != 0
        ),
    }

In [17]:
test1 = reference_add(10, 5)

print(test1)

{'result': 15, 'carry': 0, 'zero': 0, 'overflow': 0}
